## Model Selection

Before we move on we would like to choose the best possible model for each of the two cases: daily forecast and hourly forecast. It seems that for the daily forecast in the long term you choose linear_order2 in the short to medium term potentially hybrid_order2. For the hourly forecast it seems that just using XGBoost is the best possible model.

The main question is that in the daily forecast if we were to optimise the hyperparameters of the XGBoost model within the hybrid model would it make hybrid_order2 better than linear_order2 in both the short and long term. It would also be nice to do hyperparamter optimisation for the hourly forecast as well just to see whether we can improve the predictions or not. 

Finally it would be good to look at SHAP values to see if we can drop any of the features, particualarly some of the lags as they can make the models computationally expensive.

For our own use (maybe delete later): http://kaggle.com/code/prashant111/a-guide-on-xgboost-hyperparameters-tuning

https://hyperopt.github.io/hyperopt/?source=post_page

https://github.com/hyperopt/hyperopt/wiki/FMin

In [1]:
from hyperopt import hp, fmin, tpe, hp, STATUS_OK, Trials
from hyperopt.pyll import scope
import xgboost as xgb
from xgboost.callback import EarlyStopping
import pickle
import pandas as pd
from jfk_taxis import load_design, load_models, load_lags, run_forecasts, preprocess, forecast
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import TimeSeriesSplit
import time
import numpy as np

In [2]:
# First reload the significant lags
daily_lags = load_lags("daily", "eda")

hourly_lags = load_lags("hourly", "eda")

In [3]:
# Get both the full daily and hourly time series
dir_path = "../data/processed/"
df_daily = pd.read_csv(f"{dir_path}ts_daily2011-2025.csv")
df_hourly = pd.read_csv(f"{dir_path}ts_hour2011-2025.csv")

# Convert dates to datetime objects
df_daily["pickup_date"] = pd.to_datetime(df_daily["pickup_date"])
df_hourly["dt"] = pd.to_datetime(df_hourly["dt"])


In [4]:
# To pass the time series through our helper functions they need to be a pandas series indexed by a datetime object:
ts_hourly = df_hourly["trips"]
ts_hourly.index = df_hourly["dt"]

ts_daily = df_daily["trips"]
ts_daily.index = df_daily["pickup_date"]

In [5]:
# We now need to split into test and train data, we will train on the pre 2024 data and test on 2024 onwards, approx a 90:10 split
ts_daily_train = ts_daily[:"2023-12-31"]
ts_daily_test = ts_daily["2024-01-01":]

ts_hourly_train = ts_hourly[:"2023-12-31"]
ts_hourly_test = ts_hourly["2024-01-01":]

In [6]:
ts_daily_train.shape

(4748,)

In [7]:
k = ts_hourly_train.iloc[-2:]

In [8]:
type(k)

pandas.core.series.Series

In [9]:
k.iloc[0]

np.int64(313)

In [10]:
list(ts_hourly_train.iloc[-2:])

[313, 305]

In [11]:
# Define search space
space = {
    # We rely on early stopping when fitting so this isn't an optimised value
    # number of trees
    "n_estimators": 500,

    # Learning rate
    # step size shrinkage, smaller = slower but more precise learning
    "learning_rate": 0.05,

    # Depth/complexity
    # Max depth of tree, larger more complex trees but can cause overfitting
    "max_depth": scope.int(hp.quniform("max_depth", 3, 6, 1)), # scope.int ensures we take ints only
    # minimum "weight" needed in child node. Higher values more conservative, fewer splits helps prevent overfitting
    "min_child_weight": hp.loguniform("min_child_weight", -2.3, 2.3), # approx [0.1, 10]

    # Randomisation/feature subsampling
    # fraction of rows used per tree, lower adds randomness reduces overfitting
    "subsample": hp.uniform("subsample", 0.6, 1.0),
    # fraction of features used per tree
    "colsample_bytree": hp.uniform("colsample_bytree", 0.6, 1.0),

    # Regularisation
    # L2 penalty, good range is [0.1, 10] we use loguniform because this means that every order of magnitude has equal probability, the def of log uniform in hyperopt is that it returns a value exp(U(low,high)) where U is uniform dist.  
    "reg_lambda": hp.loguniform("reg_lambda", np.log(1e-2), np.log(100)), # [0.01, 100]
    # L1 penalty
    "reg_alpha": hp.loguniform("reg_alpha", np.log(1e-3), np.log(10)), # [0.001, 10]

    # Split pnealty (gamma) 
    # minimum loss reduction required to split a node, higher values = more conservative
    "gamma": hp.loguniform("gamma", -7.0, 2.3), # approx [0.0009, 10]

    "random_state": 37,
    #"early_stopping_rounds": 100,
    "eval_metric": "mae",
    "n_jobs": -1
}
    
    

In [12]:
# Function to create train and validation data for the objective function

def create_val_data(n_splits, test_size, lags, constant, order, fourier_features, time_step, ts):
    # We will store all of the folds in a dict
    fold_dict = {}
    
    # We need to split the ts using TimeSeriesSplit 
    tscv= TimeSeriesSplit(n_splits = n_splits, test_size = test_size)
    
    for fold, (train_index, test_index) in enumerate(tscv.split(ts)):
        print(f"Fold {fold}")
        print(train_index)
        # We need to preprocess the training portion of the fold
        ts_train = ts.iloc[train_index].copy()
        (X_train, y_train, dp) = preprocess(lags, constant, order, fourier_features, time_step, ts_train)

        # We don't need to preprocess the test portion of the fold because we are going to pass the deterministic process and use dp.out_sample()
        # when forecasting as we are doing a multistep forecast and need to build lags as we go. 
        y_test = ts.iloc[test_index].copy()


        # To improve memory usage set to float32
        X_train = X_train.astype("float32")
        y_train = y_train.astype("float32")
        y_test = y_test.astype("float32")
        
        fold_dict[f"fold_{fold}"] = (X_train, y_train, dp, y_test)

    return fold_dict
        

In [13]:
# Define objective function

# The objective function will take a dictionary of folds of the time series, it will forecast predictions for the test set and compute the MAE
# The loss returned will be the average MAE across the folds

def objective(space):
    model = xgb.XGBRegressor(
        n_estimators = space["n_estimators"],
        
        learning_rate = space["learning_rate"],

        max_depth = space["max_depth"],
        min_child_weight = space["min_child_weight"],

        subsample = space["subsample"],
        colsample_bytree = space["colsample_bytree"],

        reg_lambda = space["reg_lambda"],
        reg_alpha = space["reg_alpha"],

        gamma = space["gamma"],

        random_state = space["random_state"],
        #early_stopping_rounds = space["early_stopping_rounds"],
        eval_metric = space["eval_metric"],
        n_jobs = space["n_jobs"],

        # Tree method hist will eseentially bin feature values into histograms and consider 
        # and then only considers splits at bin boundaries
        tree_method = "hist"
        )


    maes = []
    for value in fold_dict.values():
        X_train = value[0]
        y_train = value[1]
        dp = value[2]
        y_test = value[3]

        print(X_train.shape)

        # We are going to use early stopping, now to avoid very long computations we will take a slight shorcut in that the validation set will have 
        # the "real lags" rather than the lags computed during the forecast on the test set of the fold. 

        # # Get the number of rows in X_train
        # num_rows = X_train.shape[0]

        # # We will validate on about 10% of the data
        # split_row = int(num_rows * 0.9) # this tells us which row to split on
        # X_val = X_train.iloc[split_row:]
        # y_val = y_train.iloc[split_row:]
    

        # X_train = X_train.iloc[:split_row]
        # y_train = y_train.iloc[:split_row]

        # Convert to numpy arrays before fitting model
        X_train_np = X_train.to_numpy(copy = True)
        y_train_np = y_train.to_numpy(copy = True)

        # X_val_np = X_val.to_numpy(copy = True)
        # y_val_np = y_val.to_numpy(copy = True)

        # Time fitting the model
        start_fit = time.time()
        # Fit the model
        model.fit(X_train_np, y_train_np)
        end_fit = time.time()

        # Time forecasting
        start_fore = time.time()
        # Run the forecast for the required steps
        y_preds = forecast(model, y_train, lags, steps, dp, None)
        end_fore = time.time()

        # Report timings
        print(f"Fit time: {end_fit - start_fit:.2f} seconds")
        print(f"Predict time: {end_fore - start_fore:.4f} seconds")

        # # See iterations stopped at
        # print("Best iteration:", model.best_iteration)
        # print("Best score:", model.best_score)

        # Compute MAE
        mae = mean_absolute_error(y_preds, y_test)
        maes.append(mae) 

    mean_mae = sum(maes) / len(maes)
    print("MAEs:", maes)
    print("Avg MAE:", mean_mae)
    return {'loss': mean_mae, 'status': STATUS_OK}

In [14]:
# Set the parameters for creat_val_data, daily ts
n_splits = 5
test_size = 365
lags = daily_lags
constant = False
order = 0
fourier_features = ["YE", "W"]
time_step = "D"
ts = ts_daily_train

# Set the parameters for objective
steps = 365

In [13]:
# Set the parameters for creat_val_data, hourly ts
n_splits = 5
test_size = 168
lags = hourly_lags[:168]
constant = False
order = 0
fourier_features = ["D", "W"]
time_step = "h"
ts = ts_hourly_train

# Set the parameters for objective
steps = 168

In [15]:
# Create the folds
fold_dict = create_val_data(n_splits, test_size, lags, constant, order, fourier_features, time_step, ts)

Fold 0
[   0    1    2 ... 2920 2921 2922]
Fold 1
[   0    1    2 ... 3285 3286 3287]
Fold 2
[   0    1    2 ... 3650 3651 3652]
Fold 3
[   0    1    2 ... 4015 4016 4017]
Fold 4
[   0    1    2 ... 4380 4381 4382]


In [ ]:
# Optimisation algorithm
trials = Trials()

best_hyperparams = fmin(fn = objective,
                        space = space,
                        algo = tpe.suggest,
                        max_evals = 100,
                        trials = trials)

(2552, 359)                                                               
Fit time: 11.47 seconds                                                   
Predict time: 0.4113 seconds                                              
(2917, 359)                                                               
Fit time: 12.06 seconds                                                   
Predict time: 0.4230 seconds                                              
(3282, 359)                                                               
Fit time: 12.46 seconds                                                   
Predict time: 0.3880 seconds                                              
(3647, 359)                                                               
Fit time: 12.68 seconds                                                   
Predict time: 0.4341 seconds                                              
(4012, 359)                                                               
Fit time: 12.85 seconds  

In [ ]:
print("The best hyperparamters are: ", "\n")
print(best_hyperparams)

In [ ]:
# It would now be interesting to use this hyperparams and use them on the forecasts from the previous notebook to see how they compare

# Load the previous non linear model
linear_models_loaded, non_linear_models_loaded = load_models("5_order_linear_daily")

# Now create our new non linear model trained on the full training data set
new_non_linear = xgb.XGBRegressor(
        n_estimators = int(best_hyperparams['n_estimators']),
        learning_rate = best_hyperparams['learning_rate'],
        subsample = best_hyperparams['subsample'],
        max_depth = int(best_hyperparams['max_depth']),
        gamma = best_hyperparams['gamma'],
        reg_alpha = best_hyperparams['reg_alpha'],
        reg_lambda = best_hyperparams['reg_lambda'],
        colsample_bytree = int(best_hyperparams['colsample_bytree']),
        min_child_weight = int(best_hyperparams['min_child_weight']),
        eval_metric = "mae")


new_non_linear.fit(X, y,
    verbose = False)

non_linear_models_loaded["new_non_linear"] = (new_non_linear, dp, None)

In [ ]:
# Steps for the forecast
steps = [1, 2, 3, 7, 14, 28, 30, 60, 180, 365, 500, 546]


In [ ]:
# Run forecasts
run_forecasts(steps, daily_lags, {}, non_linear_models_loaded, False, "D", ts_daily_train, ts_daily_test)